# FTTH QoE Risk - Process Phase (GitHub - Absolute Paths + Robust CSV)


In [ ]:
from pathlib import Path
import pandas as pd

current = Path.cwd().resolve()
ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "data").exists():
        ROOT = parent
        break
if ROOT is None:
    ROOT = current.parent if current.name == "python" else current
BASE = ROOT / "data"
PYTHON_DIR = ROOT / "python"

print(f"current: {current}")
print(f"ROOT: {ROOT}")
print(f"BASE: {BASE} existe={BASE.exists()}")

def read_csv_robust(path: Path):
    for enc in ["utf-8-sig", "utf-16", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore")

def p(fname: str) -> Path:
    return BASE / fname


In [ ]:
df_ping = read_csv_robust(p("sample1000_curr_ping.csv"))
df_udp = read_csv_robust(p("sample1000_udplatency.csv"))
df_fiber = read_csv_robust(p("fiber_units.csv"))
print(f"ping: {df_ping.shape} | udp: {df_udp.shape} | fiber: {df_fiber.shape}")
df_ping.head(3)


In [ ]:
df_ping["packet_loss_pct"] = df_ping["failures"] / (df_ping["successes"]+df_ping["failures"]) * 100
df_ping["hour"] = pd.to_datetime(df_ping["dtime"], errors='coerce').dt.hour
df_ping["is_peak_hour"] = ((df_ping["hour"]>=19) & (df_ping["hour"]<=23)).astype(int)
df_ping_fiber = df_ping.merge(df_fiber[["unit_id"]], on="unit_id", how="inner")
df_ping_fiber.head()


In [ ]:
df_udp_fiber = df_udp.merge(df_fiber[["unit_id"]], on="unit_id", how="inner")
rtt_p95 = df_udp_fiber.groupby(["unit_id","ddate"])["rtt_avg"].quantile(0.95).reset_index().rename(columns={"rtt_avg":"rtt_p95"})
df_final = df_ping_fiber.merge(rtt_p95, on=["unit_id","ddate"], how="left")
df_final["risk_flag"] = (((df_final["packet_loss_pct"]>1.5) | (df_final["rtt_p95"]>80)) & (df_final["is_peak_hour"]==1)).astype(int)
df_final.head()


In [ ]:
out_path = BASE / "risk_flag_sample.csv"
df_final.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Exportado: {out_path}")
